# **Colab Notebook with Python Code to Accompany the Paper "A Matrix Representation of Lewin Duality, and Dualizing Operators for Dual Groups of Triadic Transformations" at the Conference Mathematics and Computation in Music MCM 2026**

# **Thomas M. Fiore and Thomas Noll**

**Citation:** Fiore, Thomas M. and Thomas Noll. A Matrix Representation of Lewin Duality, and Dualizing Operators for Dual Groups of Triadic Transformations. In: Mariana Montiel, Matthew Sargent, Charles Doran, Francisco Gómez-Martín, and Gonzalo Romero-García, (eds.), *Mathematics and Computation in Music, 10th International Conference, MCM 2026, Proceedings.* Lecture Notes in Computer Science, Springer (2026)..

**Overview:** This Colab notebook contains the code to confirm Figure 1, Section 5.3, and Figure 6 of the aforementioned article. This Colab notebook is not needed to confirm the correctness of the article, since all the mathematical arguments are presented in the refereed article.

## Table of Contents

* Part 0a: Creation of Chord Formalizations and Triadic Bijection Functions
  * Package Loads
  * Store the 3 Chord Formalizations in 3 Lists and Display Them in One Data Frame
  * Create $Q_1$ and $T_1$ as Python Functions
  * Create $W_0$ and $I_7$ as Python Functions
  * Create $T_{11}$ as a Python Function
* Part 0b: Creation of Motivating Example in Figure 1, Which is the Action Table of the $TI$-Group, and Make Another Data Frame with the Columns as Specified in Figure 1, and Confirm the Two are the Same
* Part 1a: Section 5.3 Computation to Confirm that Selection of $c$-Minor as Reference Implies that Conjugation with $inv_c$ Maps $Q_1$ to $T_1$, and $W_0$ to $I_7$, and Thus the Hook Duality Operator is Realized as Conjugation with $inv_c$
  * Create the Function $inv_c$ as a Python Function from the Mentally Computed $inv_c$ Function
  * Make a Data Frame to Show the Conjugation $inv_c \circ Q_1 \circ inv_c^{-1}$ is $T_1$
  * Make a Data Frame to Show the Conjugation $inv_c \circ W_0 \circ inv_c^{-1}$ is $I_7$
* Part 1b: Section 5.3 Computation to Confirm that Selection of $C$-Major as Reference Implies that Conjugation with $inv_C$ Maps $Q_1$ to $T_{11}$, and $W_0$ to $I_7$, and Thus the Selection of $C$-Major as Reference Does NOT Realize the Hook Duality Operator as Conjugation with $inv_C$
  * Create the Function $inv_C$ as a Python Function from the Mentally Computed $inv_C$ Function
  * Make a Data Frame to Show the Conjugation $inv_C \circ Q_1 \circ inv_C^{-1}$ is $T_{11}$
  * Make a Data Frame to Show the Conjugation $inv_C \circ W_0 \circ inv_C^{-1}$ is $I_7$
* Part 2: Construction of Figure 6 in Section 6
* Make the Action Table for $\langle T_{1,5}, W_0 \rangle$ as a Data Frame, Make Another Data Frame with the Columns as Specified in Figure 6, and Confirm the Two are the Same


## Part 0a: Creation of Chord Formalizations and Triadic Bijection Functions

### Package Loads

In [116]:
import numpy as np # to store the images of functions as rows in a matrix
import pandas as pd # to better display the images of functions as columns

### Store the 3 Chord Formalizations in 3 Lists and Display Them in One Data Frame

In [117]:
# make lists of chord names
majors = ["C", "C#", "D", "E\u266d", "E", "F", "F#", "G", "A\u266d", "A", "B\u266d", "B"]
minors = ["c", "c#", "d", "e\u266d", "e", "f", "f#", "g", "a\u266d", "a", "b\u266d", "b"]
chords = majors + minors

# make lists of tuples
majors_as_tuples = [(0,4,7),
 (1,5,8),
 (2,6,9),
 (3,7,10),
 (4,8,11),
 (5,9,0),
 (6,10,1),
 (7,11,2),
 (8,0,3),
 (9,1,4),
 (10,2,5),
 (11,3,6)]
minors_as_tuples_dualistic = [(7,3,0),
                              (8,4,1),
                              (9,5,2),
                              (10,6,3),
                              (11,7,4),
                              (0,8,5),
                              (1,9,6),
                              (2,10,7),
                              (3,11,8),
                              (4,0,9),
                              (5,1,10),
                              (6,2,11)]
chords_as_tuples = majors_as_tuples + minors_as_tuples_dualistic

# make lists of (root, parity) in Hook notation
# we don't actually use Gamma, but we keep it here for future
Gamma_plus = [(0,"+"), (1,"+"), (2,"+"), (3,"+"), (4,"+"), (5,"+"),
 (6,"+"), (7,"+"), (8,"+"), (9,"+"), (10,"+"), (11,"+")]
Gamma_minus = [(0,"-"), (1,"-"), (2,"-"), (3,"-"), (4,"-"), (5,"-"),
 (6,"-"), (7,"-"), (8,"-"), (9,"-"), (10,"-"), (11,"-")]
chords_Gamma = Gamma_plus + Gamma_minus

In [118]:
# display all three formalizations in one table
pd.DataFrame({"Chord Name":chords, "PC-Seg" : chords_as_tuples, "Hook Notation":chords_Gamma})

,Chord Name,PC-Seg,Hook Notation
0,C,"(0, 4, 7)","(0, +)"
1,C#,"(1, 5, 8)","(1, +)"
2,D,"(2, 6, 9)","(2, +)"
3,E♭,"(3, 7, 10)","(3, +)"
4,E,"(4, 8, 11)","(4, +)"
5,F,"(5, 9, 0)","(5, +)"
6,F#,"(6, 10, 1)","(6, +)"
7,G,"(7, 11, 2)","(7, +)"
8,A♭,"(8, 0, 3)","(8, +)"
9,A,"(9, 1, 4)","(9, +)"


### Create $Q_1$ and $T_1$ as Python Functions

In [119]:
def Q1(x): # input is a chord name, NOT (root, parity)
  if x[0].isupper():
    return majors[(majors.index(x) + 1) % 12]
  if x[0].islower():
    return minors[(minors.index(x) - 1) % 12]

def T1(x): # input is a chord name, NOT (root, parity)
  if x[0].isupper():
    return majors[(majors.index(x) + 1) % 12]
  if x[0].islower():
    return minors[(minors.index(x) + 1) % 12]

In [120]:
# test Q1 on two elements
print(Q1("E"))
print(Q1("e"))

F
e♭


### Create $W_0$ and $I_7$ as Python Functions

In [121]:
def W0(x): # input is a chord name, NOT (root, parity)
  if x[0].isupper():
    return minors[majors.index(x)]
  if x[0].islower():
    return majors[minors.index(x)]

def I7(x): # input is a chord name, NOT (root, parity)
  # get the pcseg of input chord name
  x_as_pcseg = chords_as_tuples[ chords.index(x) ]
  # do I7 to the pcseg
  I7_of_x_as_pcseg = ( (-x_as_pcseg[0] +7)%12, (-x_as_pcseg[1] +7)%12, (-x_as_pcseg[2] +7)%12)
  # get the chord name of I7 of pcseg
  I7_of_x = chords[ chords_as_tuples.index(I7_of_x_as_pcseg) ]
  return I7_of_x

### Create $T_{11}$ as a Python Function

In [122]:
def T11(x): # input is a chord name, NOT (root, parity)
  if x[0].isupper():
    return majors[(majors.index(x) - 1) % 12]
  if x[0].islower():
    return minors[(minors.index(x) - 1) % 12]

In [123]:
# test T11 on two elements
print(T11("E"))
print(T11("e"))

E♭
e♭


# Part 0b: Creation of Motivating Example in Figure 1, Which is the Action Table of the $TI$-Group, and Make Another Data Frame with the Columns as Specified in Figure 1, and Confirm the Two are the Same

In [124]:
action_table = [chords]
for i in range(1,12):
  rowi = [T1(x) for x in action_table[i-1]]
  action_table.append(rowi)
action_table.append([I7(x) for x in chords])
for i in range(1,12):
  row12plusi = [T1(x) for x in action_table[11+i]]
  action_table.append(row12plusi)
action_table_df = pd.DataFrame(action_table)
action_table_df

,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
0,C,C#,D,E♭,E,F,F#,G,A♭,A,...,d,e♭,e,f,f#,g,a♭,a,b♭,b
1,C#,D,E♭,E,F,F#,G,A♭,A,B♭,...,e♭,e,f,f#,g,a♭,a,b♭,b,c
2,D,E♭,E,F,F#,G,A♭,A,B♭,B,...,e,f,f#,g,a♭,a,b♭,b,c,c#
3,E♭,E,F,F#,G,A♭,A,B♭,B,C,...,f,f#,g,a♭,a,b♭,b,c,c#,d
4,E,F,F#,G,A♭,A,B♭,B,C,C#,...,f#,g,a♭,a,b♭,b,c,c#,d,e♭
5,F,F#,G,A♭,A,B♭,B,C,C#,D,...,g,a♭,a,b♭,b,c,c#,d,e♭,e
6,F#,G,A♭,A,B♭,B,C,C#,D,E♭,...,a♭,a,b♭,b,c,c#,d,e♭,e,f
7,G,A♭,A,B♭,B,C,C#,D,E♭,E,...,a,b♭,b,c,c#,d,e♭,e,f,f#
8,A♭,A,B♭,B,C,C#,D,E♭,E,F,...,b♭,b,c,c#,d,e♭,e,f,f#,g
9,A,B♭,B,C,C#,D,E♭,E,F,F#,...,b,c,c#,d,e♭,e,f,f#,g,a♭


In [125]:
claimed_columns_horizontally = [chords]
for i in range(1,12):
  rowi = [Q1(x) for x in claimed_columns_horizontally[i-1]]
  claimed_columns_horizontally.append(rowi)
for i in range(0,12):
  row12plusi = [W0(x) for x in claimed_columns_horizontally[i]]
  claimed_columns_horizontally.append(row12plusi)
claimed_columns_array = np.array(claimed_columns_horizontally).T
claimed_columns_df = pd.DataFrame(claimed_columns_array)
claimed_columns_df

,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
0,C,C#,D,E♭,E,F,F#,G,A♭,A,...,d,e♭,e,f,f#,g,a♭,a,b♭,b
1,C#,D,E♭,E,F,F#,G,A♭,A,B♭,...,e♭,e,f,f#,g,a♭,a,b♭,b,c
2,D,E♭,E,F,F#,G,A♭,A,B♭,B,...,e,f,f#,g,a♭,a,b♭,b,c,c#
3,E♭,E,F,F#,G,A♭,A,B♭,B,C,...,f,f#,g,a♭,a,b♭,b,c,c#,d
4,E,F,F#,G,A♭,A,B♭,B,C,C#,...,f#,g,a♭,a,b♭,b,c,c#,d,e♭
5,F,F#,G,A♭,A,B♭,B,C,C#,D,...,g,a♭,a,b♭,b,c,c#,d,e♭,e
6,F#,G,A♭,A,B♭,B,C,C#,D,E♭,...,a♭,a,b♭,b,c,c#,d,e♭,e,f
7,G,A♭,A,B♭,B,C,C#,D,E♭,E,...,a,b♭,b,c,c#,d,e♭,e,f,f#
8,A♭,A,B♭,B,C,C#,D,E♭,E,F,...,b♭,b,c,c#,d,e♭,e,f,f#,g
9,A,B♭,B,C,C#,D,E♭,E,F,F#,...,b,c,c#,d,e♭,e,f,f#,g,a♭


In [126]:
action_table_df == claimed_columns_df

,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
0,True,True,True,True,True,True,True,True,True,True,...,True,True,True,True,True,True,True,True,True,True
1,True,True,True,True,True,True,True,True,True,True,...,True,True,True,True,True,True,True,True,True,True
2,True,True,True,True,True,True,True,True,True,True,...,True,True,True,True,True,True,True,True,True,True
3,True,True,True,True,True,True,True,True,True,True,...,True,True,True,True,True,True,True,True,True,True
4,True,True,True,True,True,True,True,True,True,True,...,True,True,True,True,True,True,True,True,True,True
5,True,True,True,True,True,True,True,True,True,True,...,True,True,True,True,True,True,True,True,True,True
6,True,True,True,True,True,True,True,True,True,True,...,True,True,True,True,True,True,True,True,True,True
7,True,True,True,True,True,True,True,True,True,True,...,True,True,True,True,True,True,True,True,True,True
8,True,True,True,True,True,True,True,True,True,True,...,True,True,True,True,True,True,True,True,True,True
9,True,True,True,True,True,True,True,True,True,True,...,True,True,True,True,True,True,True,True,True,True


In [127]:
action_table_df.equals(claimed_columns_df)

True

# Part 1a: Section 5.3 Computation to Confirm that Selection of $c$-Minor as Reference Implies that Conjugation with $inv_c$ Maps $Q_1$ to $T_1$, and $W_0$ to $I_7$, and Thus the Hook Duality Operator is Realized as Conjugation with $inv_c$

### Create the Function $inv_c$ as a Python Function from the **Mentally** Computed $inv_c$ Function

In [128]:
# make inv_c function using a look up table, and display it to check
inv_c_dictionary = {"C":"C", "C#":"C#", "D":"D", "E\u266d":"E\u266d", "E":"E", "F":"F", "F#":"F#",
                    "G":"G", "A\u266d":"A\u266d", "A":"A", "B\u266d":"B\u266d", "B":"B",
                    "c":"c", "c#":"b", "d":"b\u266d", "e\u266d":"a", "e":"a\u266d", "f":"g", "f#":"f#",
                    "g":"f", "a\u266d":"e", "a":"e\u266d", "b\u266d":"d", "b":"c#"}
def inv_c(x):
  return inv_c_dictionary[x]

inv_c_image = [inv_c(x) for x in chords]
inv_c_image_index = [chords.index(inv_c(x)) for x in chords]
pd.DataFrame({"Chord x":chords, "inv_c(x)":inv_c_image, "Output Chord Number":inv_c_image_index})

,Chord x,inv_c(x),Output Chord Number
0,C,C,0
1,C#,C#,1
2,D,D,2
3,E♭,E♭,3
4,E,E,4
5,F,F,5
6,F#,F#,6
7,G,G,7
8,A♭,A♭,8
9,A,A,9


### Make a Data Frame to Show the Conjugation $inv_c \circ Q_1 \circ inv_c^{-1}$ is $T_1$

In [129]:
# make a numpy array to contain conjugation of Q1 with inv_c, see it is T1
# here we use that inv_c has order 2!
row0 = chords
row1 = [inv_c(Q1(inv_c(x))) for x in chords]
row2 = [T1(x) for x in chords]
row3 = [chords.index(T1(x)) for x in chords]
array_for_conjugation_of_Q1 = np.array([row0, row1, row2, row3])
array_for_conjugation_of_Q1

array([['C', 'C#', 'D', 'E♭', 'E', 'F', 'F#', 'G', 'A♭', 'A', 'B♭', 'B',
        'c', 'c#', 'd', 'e♭', 'e', 'f', 'f#', 'g', 'a♭', 'a', 'b♭', 'b'],
       ['C#', 'D', 'E♭', 'E', 'F', 'F#', 'G', 'A♭', 'A', 'B♭', 'B', 'C',
        'c#', 'd', 'e♭', 'e', 'f', 'f#', 'g', 'a♭', 'a', 'b♭', 'b', 'c'],
       ['C#', 'D', 'E♭', 'E', 'F', 'F#', 'G', 'A♭', 'A', 'B♭', 'B', 'C',
        'c#', 'd', 'e♭', 'e', 'f', 'f#', 'g', 'a♭', 'a', 'b♭', 'b', 'c'],
       ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '0',
        '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23',
        '12']], dtype='<U21')

In [130]:
# create a data frame to more easily compare the conjugation of Q1 with inv_c, and T1
column_names = ['Chord x', 'inv_c(Q1(inv_c(x)))', 'T1(x)', 'Output Chord Number']
# Create and show the pandas DataFrame for the transposed matrix
df_for_conjugation_of_Q1 = pd.DataFrame(array_for_conjugation_of_Q1.T,
                         columns=column_names)
df_for_conjugation_of_Q1

,Chord x,inv_c(Q1(inv_c(x))),T1(x),Output Chord Number
0,C,C#,C#,1
1,C#,D,D,2
2,D,E♭,E♭,3
3,E♭,E,E,4
4,E,F,F,5
5,F,F#,F#,6
6,F#,G,G,7
7,G,A♭,A♭,8
8,A♭,A,A,9
9,A,B♭,B♭,10


### Make a Data Frame to Show the Conjugation $inv_c \circ W_0 \circ inv_c^{-1}$ is $I_7$

In [131]:
# make a numpy array to contain conjugation of W0 with inv_c, see it is I7
# here we use that inv_c has order 2!
row0 = chords
row1 = [inv_c(W0(inv_c(x))) for x in chords]
row2 = [I7(x) for x in chords]
row3 = [chords.index(I7(x)) for x in chords]
array_for_conjugation_of_W0 = np.array([row0, row1, row2, row3])
array_for_conjugation_of_W0

array([['C', 'C#', 'D', 'E♭', 'E', 'F', 'F#', 'G', 'A♭', 'A', 'B♭', 'B',
        'c', 'c#', 'd', 'e♭', 'e', 'f', 'f#', 'g', 'a♭', 'a', 'b♭', 'b'],
       ['c', 'b', 'b♭', 'a', 'a♭', 'g', 'f#', 'f', 'e', 'e♭', 'd', 'c#',
        'C', 'B', 'B♭', 'A', 'A♭', 'G', 'F#', 'F', 'E', 'E♭', 'D', 'C#'],
       ['c', 'b', 'b♭', 'a', 'a♭', 'g', 'f#', 'f', 'e', 'e♭', 'd', 'c#',
        'C', 'B', 'B♭', 'A', 'A♭', 'G', 'F#', 'F', 'E', 'E♭', 'D', 'C#'],
       ['12', '23', '22', '21', '20', '19', '18', '17', '16', '15', '14',
        '13', '0', '11', '10', '9', '8', '7', '6', '5', '4', '3', '2',
        '1']], dtype='<U21')

In [132]:
# create a data frame to more easily compare the conjugation of W0 with inv_c, and I7
column_names = ['Chord x', 'inv_c(W0(inv_c(x)))', 'I7(x)', 'Output Chord Number']
# Create and show the pandas DataFrame for the transposed matrix
df_for_conjugation_of_W0 = pd.DataFrame(array_for_conjugation_of_W0.T,
                         columns=column_names)
df_for_conjugation_of_W0

,Chord x,inv_c(W0(inv_c(x))),I7(x),Output Chord Number
0,C,c,c,12
1,C#,b,b,23
2,D,b♭,b♭,22
3,E♭,a,a,21
4,E,a♭,a♭,20
5,F,g,g,19
6,F#,f#,f#,18
7,G,f,f,17
8,A♭,e,e,16
9,A,e♭,e♭,15


# Part 1b: Section 5.3 Computation to Confirm that Selection of $C$-Major as Reference Implies that Conjugation with $inv_C$ Maps $Q_1$ to $T_{11}$, and $W_0$ to $I_7$, and Thus the Selection of $C$-Major as Reference Does NOT Realize the Hook Duality Operator as Conjugation with $inv_C$

### Create the Function $inv_C$ as a Python Function from the **Mentally** Computed $inv_C$ Function

In [133]:
# make inv_C function using a look up table, and display it to check
inv_C_dictionary = {"C":"C", "C#":"B", "D":"B\u266d", "E\u266d":"A", "E":"A\u266d", "F":"G", "F#":"F#",
                    "G":"F", "A\u266d":"E", "A":"E\u266d", "B\u266d":"D", "B":"C#",
                    "c":"c", "c#":"c#", "d":"d", "e\u266d":"e\u266d", "e":"e", "f":"f", "f#":"f#",
                    "g":"g", "a\u266d":"a\u266d", "a":"a", "b\u266d":"b\u266d", "b":"b"}
def inv_C(x):
  return inv_C_dictionary[x]

inv_C_image = [inv_C(x) for x in chords]
inv_C_image_index = [chords.index(inv_C(x)) for x in chords]
pd.DataFrame({"Chord x":chords, "inv_C(x)":inv_C_image, "Output Chord Number":inv_C_image_index})

,Chord x,inv_C(x),Output Chord Number
0,C,C,0
1,C#,B,11
2,D,B♭,10
3,E♭,A,9
4,E,A♭,8
5,F,G,7
6,F#,F#,6
7,G,F,5
8,A♭,E,4
9,A,E♭,3


### Make a Data Frame to Show the Conjugation $inv_C \circ Q_1 \circ inv_C^{-1}$ is $T_{11}$

In [134]:
# make a numpy array to contain conjugation of Q1 with inv_C, see it is T11
# here we use that inv_C has order 2!
row0 = chords
row1 = [inv_C(Q1(inv_C(x))) for x in chords]
row2 = [T11(x) for x in chords]
row3 = [chords.index(T11(x)) for x in chords]
array_for_conjugation_of_Q1 = np.array([row0, row1, row2, row3])
array_for_conjugation_of_Q1

array([['C', 'C#', 'D', 'E♭', 'E', 'F', 'F#', 'G', 'A♭', 'A', 'B♭', 'B',
        'c', 'c#', 'd', 'e♭', 'e', 'f', 'f#', 'g', 'a♭', 'a', 'b♭', 'b'],
       ['B', 'C', 'C#', 'D', 'E♭', 'E', 'F', 'F#', 'G', 'A♭', 'A', 'B♭',
        'b', 'c', 'c#', 'd', 'e♭', 'e', 'f', 'f#', 'g', 'a♭', 'a', 'b♭'],
       ['B', 'C', 'C#', 'D', 'E♭', 'E', 'F', 'F#', 'G', 'A♭', 'A', 'B♭',
        'b', 'c', 'c#', 'd', 'e♭', 'e', 'f', 'f#', 'g', 'a♭', 'a', 'b♭'],
       ['11', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10',
        '23', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21',
        '22']], dtype='<U21')

In [135]:
# create a data frame to more easily compare the conjugation of Q1 with inv_C, and T11
column_names = ['Chord x', 'inv_C(Q1(inv_C(x)))', 'T11(x)', 'Output Chord Number']
# Create and show the pandas DataFrame for the transposed matrix
df_for_conjugation_of_Q1 = pd.DataFrame(array_for_conjugation_of_Q1.T,
                         columns=column_names)
df_for_conjugation_of_Q1

,Chord x,inv_C(Q1(inv_C(x))),T11(x),Output Chord Number
0,C,B,B,11
1,C#,C,C,0
2,D,C#,C#,1
3,E♭,D,D,2
4,E,E♭,E♭,3
5,F,E,E,4
6,F#,F,F,5
7,G,F#,F#,6
8,A♭,G,G,7
9,A,A♭,A♭,8



### Make a Data Frame to Show the Conjugation $inv_C \circ W_0 \circ inv_C^{-1}$ is $I_7$

In [136]:
# make a numpy array to contain conjugation of W0 with inv_C, see it is I7
# here we use that inv_C has order 2!
row0 = chords
row1 = [inv_C(W0(inv_C(x))) for x in chords]
row2 = [I7(x) for x in chords]
row3 = [chords.index(I7(x)) for x in chords]
array_for_conjugation_of_W0 = np.array([row0, row1, row2, row3])
array_for_conjugation_of_W0

array([['C', 'C#', 'D', 'E♭', 'E', 'F', 'F#', 'G', 'A♭', 'A', 'B♭', 'B',
        'c', 'c#', 'd', 'e♭', 'e', 'f', 'f#', 'g', 'a♭', 'a', 'b♭', 'b'],
       ['c', 'b', 'b♭', 'a', 'a♭', 'g', 'f#', 'f', 'e', 'e♭', 'd', 'c#',
        'C', 'B', 'B♭', 'A', 'A♭', 'G', 'F#', 'F', 'E', 'E♭', 'D', 'C#'],
       ['c', 'b', 'b♭', 'a', 'a♭', 'g', 'f#', 'f', 'e', 'e♭', 'd', 'c#',
        'C', 'B', 'B♭', 'A', 'A♭', 'G', 'F#', 'F', 'E', 'E♭', 'D', 'C#'],
       ['12', '23', '22', '21', '20', '19', '18', '17', '16', '15', '14',
        '13', '0', '11', '10', '9', '8', '7', '6', '5', '4', '3', '2',
        '1']], dtype='<U21')

In [137]:
# create a data frame to more easily compare the conjugation of W0 with inv_c, and I7
column_names = ['Chord x', 'inv_C(W0(inv_C(x)))', 'I7(x)', 'Output Chord Number']
# Create and show the pandas DataFrame for the transposed matrix
df_for_conjugation_of_W0 = pd.DataFrame(array_for_conjugation_of_W0.T,
                         columns=column_names)
df_for_conjugation_of_W0

,Chord x,inv_C(W0(inv_C(x))),I7(x),Output Chord Number
0,C,c,c,12
1,C#,b,b,23
2,D,b♭,b♭,22
3,E♭,a,a,21
4,E,a♭,a♭,20
5,F,g,g,19
6,F#,f#,f#,18
7,G,f,f,17
8,A♭,e,e,16
9,A,e♭,e♭,15


# Part 2: Construction of Figure 6 in Section 6
# Make the Action Table for $\langle T_{1,5}, W_0 \rangle$ as a Data Frame, Make Another Data Frame with the Columns as Specified in Figure 6, and Confirm the Two are the Same

In [138]:
def T15(x): # input is a chord name, NOT (root, parity)
  if x[0].isupper():
    return majors[(majors.index(x) + 1) % 12]
  if x[0].islower():
    return minors[(minors.index(x) + 5) % 12]

In [139]:
action_table = [chords]
for i in range(1,12):
  rowi = [T15(x) for x in action_table[i-1]]
  action_table.append(rowi)
for i in range(0,12):
  row12plusi = [W0(x) for x in action_table[i]]
  action_table.append(row12plusi)
action_table_df = pd.DataFrame(action_table)
action_table_df

,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
0,C,C#,D,E♭,E,F,F#,G,A♭,A,...,d,e♭,e,f,f#,g,a♭,a,b♭,b
1,C#,D,E♭,E,F,F#,G,A♭,A,B♭,...,g,a♭,a,b♭,b,c,c#,d,e♭,e
2,D,E♭,E,F,F#,G,A♭,A,B♭,B,...,c,c#,d,e♭,e,f,f#,g,a♭,a
3,E♭,E,F,F#,G,A♭,A,B♭,B,C,...,f,f#,g,a♭,a,b♭,b,c,c#,d
4,E,F,F#,G,A♭,A,B♭,B,C,C#,...,b♭,b,c,c#,d,e♭,e,f,f#,g
5,F,F#,G,A♭,A,B♭,B,C,C#,D,...,e♭,e,f,f#,g,a♭,a,b♭,b,c
6,F#,G,A♭,A,B♭,B,C,C#,D,E♭,...,a♭,a,b♭,b,c,c#,d,e♭,e,f
7,G,A♭,A,B♭,B,C,C#,D,E♭,E,...,c#,d,e♭,e,f,f#,g,a♭,a,b♭
8,A♭,A,B♭,B,C,C#,D,E♭,E,F,...,f#,g,a♭,a,b♭,b,c,c#,d,e♭
9,A,B♭,B,C,C#,D,E♭,E,F,F#,...,b,c,c#,d,e♭,e,f,f#,g,a♭


In [140]:
def Tprime(i,x):  # input is chord name
  if x[0].isupper():
    return W0(majors[(5*majors.index(x) + i) % 12])
  if x[0].islower():
    return W0(minors[(5*minors.index(x) + i) % 12])

In [141]:
claimed_columns_horizontally = [chords]
for i in range(1,12):
  rowi = [T1(x) for x in claimed_columns_horizontally[i-1]]
  claimed_columns_horizontally.append(rowi)
for i in range(0,12):
  row12plusi = [Tprime(i, x) for x in chords]
  claimed_columns_horizontally.append(row12plusi)
claimed_columns_array = np.array(claimed_columns_horizontally).T
claimed_columns_df = pd.DataFrame(claimed_columns_array)
claimed_columns_df

,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
0,C,C#,D,E♭,E,F,F#,G,A♭,A,...,d,e♭,e,f,f#,g,a♭,a,b♭,b
1,C#,D,E♭,E,F,F#,G,A♭,A,B♭,...,g,a♭,a,b♭,b,c,c#,d,e♭,e
2,D,E♭,E,F,F#,G,A♭,A,B♭,B,...,c,c#,d,e♭,e,f,f#,g,a♭,a
3,E♭,E,F,F#,G,A♭,A,B♭,B,C,...,f,f#,g,a♭,a,b♭,b,c,c#,d
4,E,F,F#,G,A♭,A,B♭,B,C,C#,...,b♭,b,c,c#,d,e♭,e,f,f#,g
5,F,F#,G,A♭,A,B♭,B,C,C#,D,...,e♭,e,f,f#,g,a♭,a,b♭,b,c
6,F#,G,A♭,A,B♭,B,C,C#,D,E♭,...,a♭,a,b♭,b,c,c#,d,e♭,e,f
7,G,A♭,A,B♭,B,C,C#,D,E♭,E,...,c#,d,e♭,e,f,f#,g,a♭,a,b♭
8,A♭,A,B♭,B,C,C#,D,E♭,E,F,...,f#,g,a♭,a,b♭,b,c,c#,d,e♭
9,A,B♭,B,C,C#,D,E♭,E,F,F#,...,b,c,c#,d,e♭,e,f,f#,g,a♭


In [142]:
action_table_df == claimed_columns_df

,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
0,True,True,True,True,True,True,True,True,True,True,...,True,True,True,True,True,True,True,True,True,True
1,True,True,True,True,True,True,True,True,True,True,...,True,True,True,True,True,True,True,True,True,True
2,True,True,True,True,True,True,True,True,True,True,...,True,True,True,True,True,True,True,True,True,True
3,True,True,True,True,True,True,True,True,True,True,...,True,True,True,True,True,True,True,True,True,True
4,True,True,True,True,True,True,True,True,True,True,...,True,True,True,True,True,True,True,True,True,True
5,True,True,True,True,True,True,True,True,True,True,...,True,True,True,True,True,True,True,True,True,True
6,True,True,True,True,True,True,True,True,True,True,...,True,True,True,True,True,True,True,True,True,True
7,True,True,True,True,True,True,True,True,True,True,...,True,True,True,True,True,True,True,True,True,True
8,True,True,True,True,True,True,True,True,True,True,...,True,True,True,True,True,True,True,True,True,True
9,True,True,True,True,True,True,True,True,True,True,...,True,True,True,True,True,True,True,True,True,True


In [143]:
action_table_df.equals(claimed_columns_df)

True